In [0]:
from pyspark.sql import functions as F

landing_path = "abfss://ecom@sreeadls2797.dfs.core.windows.net/retail_sales/landing/"
bronze_table = "ecom_external_catalog.retail_sales_dev.sales_bronze"
checkpoint_path = "/Volumes/ecom_external_catalog/retail_sales_dev/pipeline_state/bronze"

source_schema = """
    sale_id STRING,
    sale_date STRING,
    product STRING,
    category STRING,
    quantity STRING,
    unit_price STRING
"""

incoming_sales = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.useManagedFileEvents", "false")
    .option("header", "true")
    .option("pathGlobFilter", "sales_day_*.csv")
    .schema(source_schema)
    .load(landing_path)
    .select("*", F.col("_metadata.file_name").alias("source_file"))
    .withColumn("ingested_at", F.current_timestamp())
)

query = (
    incoming_sales.writeStream
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
%sql
SELECT source_file, COUNT(*) AS row_count
FROM ecom_external_catalog.retail_sales_dev.sales_bronze
GROUP BY source_file
ORDER BY source_file;